# Toronto Café Location-Scoring: Café Ingest

**Pipeline stage:** Bronze (raw landing)  
**Source:** OpenStreetMap, queried via the Overpass API (`amenity=cafe`)  
**Source URL:** https://www.openstreetmap.org (data), https://overpass-api.de/api/interpreter (Overpass API)  
**Attribution:** © OpenStreetMap contributors (ODbL)  
**Destination:** MongoDB `toronto_cafe.pois`

This notebook pulls every café in the City of Toronto from OpenStreetMap and lands them in the database as the raw bronze layer. These cafés are the **competition layer** of the scoring model: for any candidate location, the number of existing cafés nearby is one of the factors that drives its score.

Each café becomes one document keyed on its OpenStreetMap id, so the load can be re-run without creating duplicates. The raw tags are kept exactly as they arrive, because every café carries a different set of them, and each document records where it came from and when it was pulled.

## 1. Setup and connection

Import the libraries, load the connection string from the local `.env`, and connect to the project database. This is the same database that holds the neighbourhoods; the cafés land in a separate collection called `pois`.

In [63]:
import os                                    # for reading the connection string from the environment
import requests                              # NEW: to call the Overpass API over the web
from datetime import datetime, timezone      # to timestamp each café at ingest
from dotenv import load_dotenv               # reads your local .env
from pymongo import MongoClient              # driver to talk to MongoDB

load_dotenv(".env")                          # load MONGODB_URI from .env
client = MongoClient(os.environ["MONGODB_URI"])   # connect to the same Atlas cluster
db = client["toronto_cafe"]                  # same project database as before

## 2. Fetch the cafés from OpenStreetMap

Send an Overpass query for every point tagged `amenity=cafe` inside a bounding box around Toronto, then check the response before reading it. The User-Agent header identifies the request, which the Overpass server requires (without it, the server refuses with a 406). The reply comes back as JSON, with the cafés in an `elements` list.

In [65]:
# Build the Overpass query as a text string (asks for café points inside a Toronto bounding box)
query = """
[out:json];
node["amenity"="cafe"](43.581,-79.639,43.855,-79.116);
out;
"""

url = "https://overpass-api.de/api/interpreter"                          # the Overpass API endpoint
headers = {"User-Agent": "toronto-cafe-scoring/1.0 (learning project)"}  # identify ourselves, or Overpass returns 406
response = requests.post(url, data={"data": query}, headers=headers)     # send the query, get a response back

response.raise_for_status()                                              # stop with a clear error if the status is not OK

data = response.json()                                                   # turn the JSON reply into a Python dict
print("cafés found:", len(data["elements"]))                            # the cafés are in the 'elements' list

cafés found: 1693


## 3. Inspect the response

Look at the top-level keys of the reply. The cafés live under `elements`; the other keys are metadata about the query and the data licence.

In [66]:
data.keys()   # peek at the top-level structure of the reply — the cafés live under 'elements'

dict_keys(['version', 'generator', 'osm3s', 'elements'])

## 4. Reshape and land into the bronze layer

Walk through every café returned. For each one, build a document keyed on its OpenStreetMap id, mark it as a café, keep its raw tags exactly as they arrive, record its position, and stamp it with its source and pull time. Writing with an upsert on the id means re-running this never creates duplicates.

Each café carries a different set of tags (some have a dozen, some have two), which is exactly why a document store is the right home for this data: every café keeps its own shape instead of being forced into fixed columns.

In [ ]:
pois = db["pois"]                                    # destination collection (same DB, separate 'pois' drawer)

for element in data["elements"]:                     # walk through every café Overpass returned

    doc = {                                          # build one café document
        "_id": element["id"],                        # OSM's unique id as our key (re-runnable, no duplicates)
        "lat": element["lat"],                       # position
        "lon": element["lon"],                       # position
        "tags": element.get("tags", {}),             # the raw messy tags, kept faithfully ({} if somehow missing)
        "poi_type": "cafe",                          # what kind of POI this is (marks it within the collection)
        "source": "osm_overpass",                    # where it came from (provenance)
        "ingested_at": datetime.now(timezone.utc),   # when we pulled it (freshness)
    }

    pois.update_one(                                 # upsert: update if this id exists, insert if new
        {"_id": doc["_id"]},                         # find any existing doc with this id
        {"$set": doc},                               # overwrite its contents with our fresh version
        upsert=True,                                 # if none exists yet, create it
    )

print("done — processed", len(data["elements"]), "elements")   # feedback